In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.options.display.max_columns=1000

## Объединим файлы

In [ ]:
df_train = pd.concat((
    pd.read_parquet(f"aggregated/batch_{i}.parquet")
    for i in range(1, 20 + 1)
))

df_test = pd.concat((
    pd.read_parquet(f"aggregated/batch_{i}.parquet")
    for i in range(21, 25 + 1)
))

## Посмотрим признак

In [ ]:
feature = (df_train["StartCentipawnLoss15"] // 10).clip(0, 25)

In [ ]:
fig = px.line(
    feature.value_counts().sort_index()
)

fig.data[0].mode = "lines+markers"
fig.update_layout(
    template="plotly_white",
    showlegend=False
)

fig.show()

In [ ]:
fig = px.line(
    df_train.groupby(feature).agg({"Elo": "mean"})
)

fig.data[0].mode = "lines+markers"
fig.update_layout(
    template="plotly_white",
    showlegend=False
)

fig.show()

## Сохраняем выводы

In [ ]:
def bin_features(df):
    """Делает разбиение на бины (для уменьшения переобучения)"""
    
    # Ошибки
    df["NBlunbers"] = (df["NBlunbers"]).clip(0, 20)
    df["NOkayMoves"] = (df["NOkayMoves"] // 5).clip(0, 30)
    df["MeanBlunbers"] = (df["MeanBlunbers"] // 0.02).clip(0, 15)
    df["MeanMistakes"] = (df["MeanMistakes"] // 0.05).clip(0, 10)
    df["MeanBadMoves"] = (df["MeanBadMoves"] // 0.02).clip(0, 24)
    df["MeanOkayMoves"] = (df["MeanOkayMoves"] // 0.02).clip(20, 50)
    
    # Средний ход ошибок
    df["MoveNumberBlunder"] = (df["MoveNumberBlunder"] // 2).clip(0, 25)
    df["MoveNumberMistake"] = (df["MoveNumberMistake"] // 3).clip(0, 15)
    df["MoveNumberBadMove"] = (df["MoveNumberBadMove"] // 3).clip(0, 15)
    
    # Eval
    df["MeanAbsEval"] = (df["MeanAbsEval"] // 20).clip(0, 40)
    df["EvalStd"] = (df["EvalStd"] // 50).clip(0, 18)
    df["NEqualGame300"] = (df["NEqualGame300"] // 3).clip(0, 30)
    df["MeanLostGame600"] = (df["MeanLostGame600"] // 0.05).clip(0, 18)
    
    # Потери сантипешек
    df["MeanCentipawnLoss"] = (df["MeanCentipawnLoss"] // 10).clip(0, 22)
    df["StartCentipawnLoss15"] = (df["StartCentipawnLoss15"] // 10).clip(0, 25)
    df["KnightCentipawnLoss"] = (df["KnightCentipawnLoss"] // 20).clip(0, 16)
    df["PawnCentipawnLoss"] = (df["PawnCentipawnLoss"] // 10).clip(1, 20)
    
    # Прочее
    df["MeanHasMate"] = (df["MeanHasMate"] // 0.05).clip(0, 8)
    df["MeanChecks"] = (df["MeanChecks"] // 0.02).clip(0, 12)
    df["NMoves"] = (df["NMoves"] // 5).clip(0, 20)
    
    return df

In [ ]:
bin_features(df_train).to_parquet("datasets/binned_train.parquet")
bin_features(df_test).to_parquet("datasets/binned_test.parquet")